[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mionoyui/RFD4-Proteina-Tutorial/blob/main/RFD4_Proteina_Binder_Design_Tutorial.ipynb)


# RFD4-Proteina で学ぶバインダー設計（日本語チュートリアル・T4対応）

標的タンパク質の表面に結合する小さなタンパク質（バインダー）を
[RFD4-Proteina](https://github.com/RosettaCommons/RFD4-Proteina) で設計します。
RFD4-Proteina はフローマッチングにもとづく生成モデルで、配列と全原子構造を一度に作ります。

## 題材は PD-1

標的は免疫チェックポイント受容体 PD-1 の細胞外ドメイン（PDB ID: 4ZQK の B 鎖）です。
T細胞上の PD-1 が腫瘍側の PD-L1 と結合すると、T細胞の働きが抑えられることが知られています。
ニボルマブやペムブロリズマブはこの経路を阻害する抗体医薬です。

4ZQK は PD-1 と PD-L1 の複合体で、A 鎖が PD-L1、B 鎖が PD-1 です。
このチュートリアルでは PD-L1（A 鎖）を外し、空いた結合面を狙ってバインダーを設計します。
PD-L1 の代わりに PD-1 へ結合する人工タンパク質を作ることが目標です。
なお B 鎖には結晶化のための C93S 変異が入っており、遊離のシステインを持ちません。

B 鎖の配列長は 118 残基で、座標が観測されているのは 106 残基です。
16GB の T4 でも 1 本あたり数分で生成が終わる規模です。

## このノートブックが扱う範囲

構造と配列の生成、および構造だけで計算できる指標（原子の衝突、結合長、ホットスポット遵守など）までです。
AlphaFold で折り畳みを予測し直す再検証は Apptainer イメージを必要とし、Colab では動きません。
よって、ここで得られる設計はすべて未検証の候補です。
ステップ 11 で書き出す FASTA を AlphaFold3 や OpenFold3 のノートブックに渡し、
界面の信頼度と設計モデルへの RMSD の両方で絞り込んでください。
界面の信頼度だけでは、別の向きで結合した構造を高く評価してしまいます。

## 所要時間の目安

T4、標的 106 残基、バインダー 55-85 残基、設計 4 本の場合です。

| 工程 | 時間 |
|---|---|
| ステップ3 インストール | 3-6 分 |
| ステップ4 重みのダウンロード | 2-4 分（セッションごと） |
| ステップ8 生成 | 1 本あたり 2-5 分 |

## 実行前の準備

[ランタイム] → [ランタイムのタイプを変更] で GPU を選び、上から順にセルを実行してください。
既定値は T4 を想定しています。A100 や L4 が割り当てられた場合は、ステップ 1 が自動でより高い設定を提案します。

---
## ステップ1 · GPU の確認と自動設定

割り当てられた GPU を読み取り、後のセルが使う 3 つの値を決めます。
インストールする wheel の種類（CUDA 12 用か CUDA 13 用か）、計算精度、融合カーネルを使うかどうかです。

bfloat16 は Ampere 世代（sm_80）以降でしか動きません。
T4 は sm_75 なので fp16 に落とし、cuEquivariance の融合カーネルも標準の PyTorch 実装に差し替えます。
このセルはバインダー長や思考量の推奨値も決めます。

In [ ]:
#@title 1. GPU の確認と自動設定
import re, subprocess

def _query(fields):
    cmd = ["nvidia-smi", "--query-gpu=" + fields, "--format=csv,noheader"]
    return subprocess.run(cmd, capture_output=True, text=True).stdout.strip().splitlines()

_rows = _query("name,compute_cap,memory.total,driver_version")
if not _rows or not _rows[0].strip():
    raise SystemExit(
        "GPU が見つかりません。\n"
        "[ランタイム] → [ランタイムのタイプを変更] で GPU を選び、このセルを再実行してください。")

GPU_NAME, _cap, _mem, GPU_DRIVER = [x.strip() for x in _rows[0].split(",")]
GPU_SM     = int(round(float(_cap) * 10))
GPU_MEM_GB = round(int(_mem.split()[0]) / 1024, 1)

# ドライバの CUDA バージョンで wheel のビルドを選ぶ。nvidia-smi のヘッダに出ている。
_banner = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
_hit = re.search(r"CUDA Version:\s*([0-9]+\.[0-9]+)", _banner)
DRIVER_CUDA = float(_hit.group(1)) if _hit else 12.0

CUDA_EXTRA = "all-cuda13" if DRIVER_CUDA >= 13.0 else "all-cuda12"
MODERN_GPU = GPU_SM >= 80                      # Ampere 以降かどうか

PRECISION = "bf16-mixed" if MODERN_GPU else "16-mixed"
KERNEL_OVERRIDES = [] if MODERN_GPU else ["nn.tri_mult_backend=torch", "nn.attention_type=torch"]

# 以降のセルが参照する推奨値
if GPU_MEM_GB >= 38:
    SUGGEST = {"recycle": "thinking_medium", "binder": (60, 110), "designs": 8, "batch": 2}
elif GPU_MEM_GB >= 22:
    SUGGEST = {"recycle": "thinking_medium", "binder": (60, 100), "designs": 6, "batch": 1}
else:
    SUGGEST = {"recycle": "thinking_low", "binder": (55, 85), "designs": 4, "batch": 1}

print("GPU          :", GPU_NAME, f"({GPU_MEM_GB} GB, sm_{GPU_SM})")
print("ドライバ     :", GPU_DRIVER, f"(CUDA {DRIVER_CUDA})")
print("wheel        :", f"rfd4-proteina[{CUDA_EXTRA}]")
print("計算精度     :", PRECISION, "" if MODERN_GPU else "（bfloat16 非対応の世代なので fp16）")
print("カーネル     :", "cuEquivariance（融合）" if MODERN_GPU else "PyTorch 標準（フォールバック）")
print()
print("このGPUでの推奨値")
print("  バインダー長 :", f"{SUGGEST['binder'][0]}-{SUGGEST['binder'][1]} 残基", "（ステップ7）")
print("  思考量       :", SUGGEST["recycle"], "（ステップ8）")
print("  設計本数     :", SUGGEST["designs"], f"/ バッチ {SUGGEST['batch']}", "（ステップ8）")

if GPU_SM >= 100:
    print("\n参考: Blackwell 世代です。融合カーネルで落ちる場合は、ステップ8の EXTRA_OVERRIDES に")
    print("      nn.tri_mult_backend=torch nn.attention_type=torch を入れてください。")

---
## ステップ2 · 保存先の決定（Google Drive の接続は基本的に不要）

既定では Drive には接続せず、出力はローカルディスクの `/content` 以下に書かれます。
セッション終了とともに消えるため、ステップ 12 で ZIP にまとめてダウンロードしてください。

Colab は無操作が続くと切断され、ランタイムには時間の上限もあります。
切断をまたいで同じ実験を続けたい場合だけ、下の `BASE_DIR` を Drive 上のパス
（`/content/drive/MyDrive/...`）に変更してください。接続すると、切断後に同じ `RUN_TAG` で
もう一度実行するだけで、足りない分だけ追加生成できます。
生成済みの設計は `skip_existing=true` により再生成されません。

`RUN_TAG` は実験の名前です。条件を変えて試すときは名前を変えると、結果が混ざりません。

モデルの重みは Drive に置かず、ローカルディスクの `/content/hf_cache` に毎回ダウンロードします。
重みは safetensors 形式で、読み込みに mmap を使います。
Drive は FUSE 経由のファイルシステムなので、mmap による読み出しが壊れ、
`SafetensorError: header too large` で停止することがあります。
ダウンロードは 2-4 分で、Drive からの読み出しと大差ありません。

In [ ]:
#@title 2. 保存先と実験名
import os

BASE_DIR = "/content/rfd4_binder_tutorial"  #@param {type:"string"}
#  BASE_DIR: /content/drive/MyDrive/... を指定した場合のみ Drive を接続します。
RUN_TAG  = "pd1_binder_01"  #@param {type:"string"}
#  RUN_TAG: 実験名。同じ名前で再実行すると、その実験に設計を追加します。
#           条件を変えるときは新しい名前にしてください。

# Drive 上のパスを指定したときだけ接続する。ふだんは認証の手間をかけない。
if BASE_DIR.startswith("/content/drive"):
    if os.path.ismount("/content/drive"):
        print("Drive はすでに接続済みです。")
    else:
        from google.colab import drive
        drive.mount("/content/drive")

# 重みはローカルディスクに置く。Drive（FUSE）では safetensors の mmap 読み出しが壊れる。
HF_HOME = "/content/hf_cache"
os.environ["HF_HOME"] = HF_HOME
RUN_DIR   = os.path.join(BASE_DIR, "runs", RUN_TAG)
INPUT_DIR = os.path.join(RUN_DIR, "inputs")
for _d in (HF_HOME, INPUT_DIR):
    os.makedirs(_d, exist_ok=True)

_existing = os.path.isdir(os.path.join(RUN_DIR, "inference_samples"))

print("\n実験名       :", RUN_TAG)
print("出力先       :", RUN_DIR, "（既存の実験に追加します）" if _existing else "（新規）")
print("重みの保存先 :", HF_HOME, "（ローカルディスク）")
if not BASE_DIR.startswith("/content/drive"):
    print("\n注意: ローカルディスクに保存します。セッションが切れると出力は消えます。")
    print("      ステップ12 で ZIP にまとめてダウンロードしてください。")
    print("      切断をまたいで作業する場合は BASE_DIR に Drive のパスを指定してください。")

---
## ステップ3 · RFD4-Proteina のインストール

wheel は Python 3.12 系を要求し、CUDA のライブラリを含めて約 7GB になります。
Colab 本体の Python に入れると依存関係がぶつかるため、`uv` で専用の仮想環境を `/content/rfd4_env` に作り、
そこへインストールします。解析と可視化のセルは Colab 本体の Python を使い続けます。

仮想環境は同じセッション内で再利用されるので、2 回目以降の実行は数秒で終わります。
入れ直したいときは `FORCE_REINSTALL` をオンにしてください。

In [ ]:
#@title 3. インストール（初回は 3-6 分）
import os, sys, shutil, subprocess, time, importlib.util

FORCE_REINSTALL = False  #@param {type:"boolean"}

ENV_DIR  = "/content/rfd4_env"
RFD4_PY  = f"{ENV_DIR}/bin/python"
GEN_BIN  = f"{ENV_DIR}/bin/rfd4-generate"

def run(cmd):
    print("+", cmd, flush=True)
    if subprocess.call(cmd, shell=True) != 0:
        raise RuntimeError("失敗したコマンド: " + cmd)

if FORCE_REINSTALL and os.path.isdir(ENV_DIR):
    shutil.rmtree(ENV_DIR)

_t0 = time.time()

if os.path.exists(GEN_BIN):
    print("仮想環境はすでにあります。インストールを省略します。")
else:
    if shutil.which("uv") is None:
        run(f"{sys.executable} -m pip -q install uv")
    run(f"uv venv --python 3.12 {ENV_DIR}")
    # --python でインタプリタを明示する。環境変数 VIRTUAL_ENV だけでは
    # Colab 本体の Python に入ってしまい、しかも終了コードは 0 になる。
    run(f'uv pip install --python {RFD4_PY} --torch-backend=auto '
        f'"rfd4-proteina[{globals().get("CUDA_EXTRA", "all-cuda12")}]"')

# 解析側（Colab 本体の Python）で使うライブラリ
for _mod, _pkg in (("gemmi", "gemmi"), ("py3Dmol", "py3Dmol"), ("yaml", "pyyaml")):
    if importlib.util.find_spec(_mod) is None:
        run(f"{sys.executable} -m pip -q install {_pkg}")

# 仮想環境を呼び出すときの環境変数。Colab は MPLBACKEND に
# matplotlib_inline のバックエンドを設定しており、仮想環境内には存在しないため、
# matplotlib の import が失敗して rfproteina 全体が読み込めなくなる。
RFD4_ENV = {k: v for k, v in os.environ.items()
            if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV")}
RFD4_ENV["MPLBACKEND"] = "Agg"

_probe = subprocess.run(
    [RFD4_PY, "-c", "import sys, rfproteina, rfd4_proteina as m;"
                    "print(sys.version.split()[0], getattr(m, '__version__', '?'))"],
    env=RFD4_ENV, capture_output=True, text=True)

if _probe.returncode != 0:
    _lst = subprocess.run(f"uv pip list --python {RFD4_PY}", shell=True,
                          capture_output=True, text=True).stdout
    raise RuntimeError(
        "仮想環境の読み込み確認に失敗しました。\n"
        f"--- エラー ---\n{_probe.stderr[-900:]}\n"
        f"--- インストール済み ---\n{_lst[-600:]}\n"
        "一覧が空の場合は、インストール先が違います。FORCE_REINSTALL=True で作り直してください。")

_py, _ver = _probe.stdout.split()
print(f"\nrfd4-generate : {GEN_BIN}")
print(f"Python        : {_py}")
print(f"rfd4-proteina : {_ver}")
print(f"所要時間      : {time.time() - _t0:.0f} 秒")

---
## ステップ4 · モデルの重みの取得と検証

公開されている重み（生成モデルと自己符号化器、合計約 1.7GB）をローカルディスクに落とします。
公開リポジトリなので認証トークンは不要です。

取得したあと、safetensors のヘッダを実際に開いて読めるかどうかを確かめます。
壊れたファイルがキャッシュに残っていると、ステップ8 がモデルの読み込み中に
`SafetensorError: header too large` で止まります。
生成を始めてから気づくと待ち時間が無駄になるので、ここで検証します。
失敗した場合はキャッシュを消して 1 回だけ取り直します。

In [ ]:
#@title 4. 重みの取得と検証
import os, glob, shutil, stat, subprocess, time

HF_REPO          = "nvidia/RFD4-Proteina"  #@param {type:"string"}
FORCE_REDOWNLOAD = False  #@param {type:"boolean"}
#  FORCE_REDOWNLOAD: キャッシュを消してから取り直します。

# このリポジトリにはルートの重み以外に別系統のモデルやアダプタも入っており、
# 全部取ると 100GB を超えてディスクが埋まる。生成が読むのはルートの 2 ファイルだけ。
WEIGHT_PATTERNS = ["config.json", "model.safetensors"]

HF_HOME = globals().get("HF_HOME", "/content/hf_cache")
assert not HF_HOME.startswith("/content/drive"), (
    "重みを Drive に置くと safetensors の mmap 読み出しが壊れます。ステップ2 を実行し直してください。")
os.makedirs(HF_HOME, exist_ok=True)
os.environ["HF_HOME"] = HF_HOME

RFD4_PY   = globals().get("RFD4_PY", "python")
_env      = dict(globals().get("RFD4_ENV", os.environ)); _env["HF_HOME"] = HF_HOME
REPO_DIR  = os.path.join(HF_HOME, "hub", "models--" + HF_REPO.replace("/", "--"))


def _root_weights():
    """ルート直下の model.safetensors だけを返す（下層のアダプタは対象外）。"""
    return sorted(glob.glob(REPO_DIR + "/snapshots/*/model.safetensors"))


def _cache_size():
    return sum(os.path.getsize(q) for q in glob.glob(REPO_DIR + "/blobs/*")
               if os.path.isfile(q))


def _dir_size(path):
    """du と同じ数え方。HF キャッシュは blobs の実体を snapshots から参照するので、
    リンクを辿ると同じ中身を何重にも数えてしまう。"""
    seen, total = set(), 0
    for r, _d, fs in os.walk(path):
        for f in fs:
            try:
                st = os.lstat(os.path.join(r, f))
            except OSError:
                continue
            if stat.S_ISLNK(st.st_mode):
                continue                      # シンボリックリンクは実体を数えない
            key = (st.st_dev, st.st_ino)
            if key in seen:
                continue                      # ハードリンクは 1 回だけ数える
            seen.add(key)
            total += st.st_size
    return total


def _verify_weights():
    """生成時と同じ仮想環境で safetensors のヘッダを開いて確かめる。"""
    files = _root_weights()
    if not files:
        return False, "model.safetensors が見つかりません"
    script = (
        "from safetensors import safe_open\n"
        f"for f in {files!r}:\n"
        "    with safe_open(f, framework='pt') as h:\n"
        "        n = len(h.keys())\n"
        "    print('OK', f.split('/')[-1], n, 'tensors')\n")
    r = subprocess.run([RFD4_PY, "-"], input=script, env=_env,
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout if r.returncode == 0 else r.stderr)[-600:]


if FORCE_REDOWNLOAD:
    shutil.rmtree(REPO_DIR, ignore_errors=True)

for _attempt in (1, 2):
    if not _root_weights():
        _free = shutil.disk_usage("/content").free / 1e9
        assert _free > 5, (f"ディスクの空きが {_free:.1f} GB しかありません。"
                           "不要なファイルを削除してから実行してください。")
        print(f"\nダウンロード中: {HF_REPO} の {', '.join(WEIGHT_PATTERNS)}（約 1.8 GB）",
              flush=True)
        _t0 = time.time()
        # 出力を捕まえずそのまま流す。進捗が見えないと、止まっているのか
        # 遅いだけなのか判断できない。
        _r = subprocess.run(
            [RFD4_PY, "-c", "from huggingface_hub import snapshot_download as d;"
                            f" d({HF_REPO!r}, allow_patterns={WEIGHT_PATTERNS!r})"],
            env=_env)
        assert _r.returncode == 0, "ダウンロードに失敗しました。このセルを再実行してください。"
        _sec, _got = time.time() - _t0, _cache_size()
        print(f"\n  {_sec:.0f} 秒 / {_got / 1e9:.2f} GB"
              f"（{_got / 1e6 / max(_sec, 1):.1f} MB/s）")

    _ok, _msg = _verify_weights()
    if _ok:
        print("検証:")
        for _line in _msg.strip().splitlines():
            print("  ", _line)
        break
    print("\n検証に失敗しました。キャッシュを削除して取り直します。")
    print("  ", _msg.strip().splitlines()[-1] if _msg.strip() else "（出力なし）")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
else:
    raise RuntimeError(
        "重みの検証に 2 回失敗しました。ディスクの空き容量を確認してください（3GB 以上）。"
        "解決しない場合は FORCE_REDOWNLOAD をオンにして、もう一度実行してください。")

_size = _dir_size(HF_HOME)
print(f"\nHF_HOME : {HF_HOME}")
print(f"サイズ  : {_size / 1e9:.2f} GB（想定 1.8 GB 前後）")
for _name in sorted(os.listdir(HF_HOME)):
    _path = os.path.join(HF_HOME, _name)
    if os.path.isdir(_path):
        print(f"  {_name + '/':<8} {_dir_size(_path) / 1e9:.2f} GB")
print(f"空き    : {shutil.disk_usage('/content').free / 1e9:.0f} GB")

if _size > 5e9:
    # リポジトリ全体を引いてしまった場合。個別のサブディレクトリを消すのではなく、
    # HF_HOME ごと作り直すのが確実。
    print("\n注意: 想定より大きいです。次の 2 行を新しいセルで実行してから、このセルをやり直してください。")
    print("  import shutil")
    print(f"  shutil.rmtree({HF_HOME!r})")

---
## ステップ5 · 標的構造の準備

RFD4-Proteina が読むのは `.cif` 系の形式だけで、`.pdb` は受け付けません。
このセルは RCSB から構造を取得し、`gemmi` で次の整理をしてから mmCIF として書き出します。

- 別配座、水素、水分子、リガンドを除く
- 1 番目のモデルだけを残す
- `KEEP_CHAINS` で指定した鎖だけを残す
- `label_seq_id` を振り直す（ステップ7 の contig がこの番号を使います）

既定値の `4ZQK` と `KEEP_CHAINS="B"` は、PD-1 だけを残して PD-L1（A 鎖）を外す指定です。
外した鎖の名前は記録され、ステップ6 がホットスポットを選ぶときに使います。

計算量は残基数の合計で決まります。大きな標的を使う場合は、設計したい領域だけに切り詰めてください。
T4 で安全に収まるのは、標的と設計部分の合計で 200 残基程度までです。
手元の構造を使う場合は `SOURCE` を `upload` か `path` に変えてください。

In [ ]:
#@title 5. 標的構造の読み込みと整理
import os, subprocess, gemmi

SOURCE       = "fetch_pdb"  #@param ["fetch_pdb", "upload", "path"]
PDB_ID       = "4ZQK"       #@param {type:"string"}
LOCAL_PATH   = ""           #@param {type:"string"}
KEEP_CHAINS  = "B"          #@param {type:"string"}
#  KEEP_CHAINS: 残す鎖をカンマ区切りで指定します。空欄ならすべて残します。
DROP_LIGANDS = True         #@param {type:"boolean"}

RUN_DIR   = globals().get("RUN_DIR", "/content/run")
INPUT_DIR = os.path.join(RUN_DIR, "inputs")
os.makedirs(INPUT_DIR, exist_ok=True)

if SOURCE == "fetch_pdb":
    raw = f"{INPUT_DIR}/{PDB_ID.lower()}_raw.cif"
    if not os.path.exists(raw):
        subprocess.run(f'curl -sSL -f -o "{raw}" '
                       f'https://files.rcsb.org/download/{PDB_ID.upper()}.cif',
                       shell=True, check=True)
    assert os.path.getsize(raw) > 2000, "RCSB から構造が返っていません。PDB ID を確認してください。"
elif SOURCE == "upload":
    from google.colab import files
    _up = files.upload()
    _name = list(_up)[0]
    raw = f"{INPUT_DIR}/{_name}"
    with open(raw, "wb") as fh:
        fh.write(_up[_name])
else:
    raw = LOCAL_PATH.strip()
    assert raw and os.path.exists(raw), f"ファイルが見つかりません: {raw!r}"

st = gemmi.read_structure(raw)
st.setup_entities()
st.remove_alternative_conformations()
st.remove_hydrogens()
if DROP_LIGANDS:
    st.remove_ligands_and_waters()
while len(st) > 1:
    del st[1]                                   # 1番目のモデルだけ残す

RAW_CIF = raw                                   # ステップ6 が相手鎖を読むために使う
DROPPED_CHAINS = []
keep = [c.strip() for c in KEEP_CHAINS.split(",") if c.strip()]
if keep:
    present = {c.name for c in st[0]}
    missing = [c for c in keep if c not in present]
    assert not missing, f"鎖 {missing} がありません。この構造の鎖は {sorted(present)} です。"
    DROPPED_CHAINS = [c.name for c in st[0] if c.name not in keep]
    for name in DROPPED_CHAINS:
        st[0].remove_chain(name)

st.setup_entities()
st.assign_label_seq_id()                        # contig が使う番号を確定させる

# 連番情報を持たない入力（SEQRES のない PDB など）では番号が入らないので、鎖ごとに 1 から振る。
if any(r.label_seq is None for ch in st[0] for r in ch):
    for ch in st[0]:
        for _i, r in enumerate(ch, start=1):
            r.label_seq = _i
    print("注意: 入力に連番情報がないため、label 番号を鎖ごとに 1 から振り直しました。\n")

TARGET_CIF = os.path.join(INPUT_DIR, "target.cif")
st.make_mmcif_document().write_file(TARGET_CIF)

TARGET_CHAIN_SEQ = {}
_total = 0
print("標的を書き出しました:", TARGET_CIF, "\n")
for ch in st[0]:
    res = list(ch)
    _total += len(res)
    _seq = gemmi.one_letter_code([r.name for r in res]).upper()
    TARGET_CHAIN_SEQ[ch.name] = _seq
    print(f"  鎖 {ch.name}: {len(res):>4} 残基   "
          f"著者番号 {res[0].seqid.num}-{res[-1].seqid.num}   "
          f"label 番号 {res[0].label_seq}-{res[-1].label_seq}")

if DROPPED_CHAINS:
    print("\n  外した鎖:", ", ".join(DROPPED_CHAINS),
          "（ステップ6 がこの鎖との接触からホットスポットを選びます）")
print(f"\n  合計 {_total} 残基")
if _total > 250:
    print("  標的が大きめです。T4 では設計部分と合わせて 200 残基以内に収めると安全です。")

---
## ステップ6 · ホットスポットの指定

バインダーに狙わせたい残基をホットスポットと呼びます。ここで最もつまずきやすいのが残基番号です。
contig と `sel(...)` が参照するのは mmCIF の `label_seq_id`、つまり 1 から連番で振り直した番号で、
PDB のページに表示される著者番号ではありません。
ステップ 5 で番号を振り直したので、両者はふつう一定の差になりますが、欠損があるとずれ方が変わります。

既定の `複合体から自動抽出` は、ステップ5 で外した鎖との接触からホットスポットを決めます。
元の複合体に戻って、標的鎖の各残基が相手鎖の原子と 4.5 Å 以内で何回接していたかを数え、
多い順に、互いに 7 Å 以上離れた残基を選びます。
4ZQK なら PD-L1 が実際に触っていた面が選ばれるので、残基番号を自分で調べる必要がありません。
離れた残基を選ぶのは、固まった指定だと狙いが一点に寄ってしまうためです。

接触の一覧も表示されるので、別の面を狙いたい場合は番号を見て
`手動で番号を指定` に切り替えてください。単量体の構造を使う場合も手動指定になります。
どちらの場合も、受け取った著者番号を `label_seq_id` に変換して選択式を組み立てます。
`HOTSPOT_LEVEL` は狙う粒度を決めます。

- `sidechain_tip` は側鎖の中ほどと先端の 2 原子を指定します。残基のどの面に接してほしいかを伝えられます。
- `residue` は残基の全原子を指定します。条件は緩くなりますが、側鎖ではなく主鎖に接するだけで条件を満たす設計も出ます。

ホットスポットは強制ではなく誘導です。守られたかどうかは、ステップ 9 の `hotspot_conditioning` 系の指標で確認します。

In [ ]:
#@title 6. ホットスポットの決定
import gemmi

TARGET_CHAIN  = "B"  #@param {type:"string"}
HOTSPOT_MODE  = "複合体から自動抽出"  #@param ["複合体から自動抽出", "手動で番号を指定", "指定しない"]
HOTSPOTS_AUTH = ""   #@param {type:"string"}
#  HOTSPOTS_AUTH: 手動指定のときだけ使います。著者番号をカンマ区切りで書きます。
N_HOTSPOTS    = 3    #@param {type:"slider", min:1, max:6, step:1}
HOTSPOT_LEVEL = "sidechain_tip"  #@param ["sidechain_tip", "residue"]
SHOW_CONTACTS = True  #@param {type:"boolean"}

# 残基種ごとの「側鎖の中ほど」と「先端」。どちら向きの面を使わせたいかを表す。
TIP_ATOMS = {
    "ALA": ["CB"],         "ARG": ["CD", "NH1"], "ASN": ["CG", "ND2"],
    "ASP": ["CG", "OD1"],  "CYS": ["CB", "SG"],  "GLN": ["CD", "NE2"],
    "GLU": ["CD", "OE1"],  "GLY": ["CA"],        "HIS": ["CG", "NE2"],
    "ILE": ["CG1", "CD1"], "LEU": ["CG", "CD1"], "LYS": ["CD", "NZ"],
    "MET": ["CG", "SD"],   "PHE": ["CG", "CZ"],  "PRO": ["CG"],
    "SER": ["CB", "OG"],   "THR": ["CB", "OG1"], "TRP": ["CD2", "CZ3"],
    "TYR": ["CG", "OH"],   "VAL": ["CB", "CG1"],
}

TARGET_CIF = globals().get("TARGET_CIF")
assert TARGET_CIF, "先にステップ5 を実行してください。"

_st = gemmi.read_structure(TARGET_CIF); _st.setup_entities()
_ch = _st[0][TARGET_CHAIN]
assert _ch is not None, f"鎖 {TARGET_CHAIN} がありません。鎖は {[c.name for c in _st[0]]} です。"

auth2label = {r.seqid.num: r.label_seq for r in _ch if r.label_seq is not None}
by_label   = {r.label_seq: r for r in _ch if r.label_seq is not None}
by_auth    = {r.seqid.num: r for r in _ch if r.label_seq is not None}
assert auth2label, "label 番号が取れません。ステップ5 をもう一度実行してください。"
_gaps = sorted({lab - auth for auth, lab in auth2label.items()})

print(f"鎖 {TARGET_CHAIN}: {len(auth2label)} 残基")
print("label - 著者番号の差:", _gaps[0] if len(_gaps) == 1 else f"{_gaps} （一定ではありません）")


def _one(res):
    return gemmi.find_tabulated_residue(res.name).one_letter_code.upper()


def _contacts_with_partners(cutoff=4.5):
    """元の複合体で、標的鎖が相手鎖と接していた残基を数える。"""
    raw = globals().get("RAW_CIF")
    partners = globals().get("DROPPED_CHAINS") or []
    if not raw or not partners:
        return {}
    st = gemmi.read_structure(raw)
    st.setup_entities(); st.remove_hydrogens(); st.remove_ligands_and_waters()
    while len(st) > 1:
        del st[1]
    # 結晶格子を外す。残すと対称の写しを接触として数えてしまう。
    st.cell = gemmi.UnitCell(); st.spacegroup_hm = "P 1"; st.setup_cell_images()
    if TARGET_CHAIN not in [c.name for c in st[0]]:
        return {}
    ns = gemmi.NeighborSearch(st, cutoff).populate()
    want = set(partners)
    counts = {}
    for res in st[0][TARGET_CHAIN]:
        n = 0
        for atom in res:
            for mark in ns.find_atoms(atom.pos, "\0", radius=cutoff):
                if st[0][mark.chain_idx].name in want:
                    n += 1
        if n:
            counts[res.seqid.num] = n
    return counts


def _spread_out(ranked, limit, min_dist=7.0):
    """接触数の多い順に、互いに離れた残基を選ぶ。固まると狙いが一点に寄る。"""
    picked, pos = [], []
    for num in ranked:
        res = by_auth.get(num)
        if res is None:
            continue
        at = res.find_atom("CB", "*") or res.find_atom("CA", "*")
        if at is None:
            continue
        if all(at.pos.dist(p) > min_dist for p in pos):
            picked.append(num); pos.append(at.pos)
        if len(picked) >= limit:
            break
    return picked


_wanted = []
if HOTSPOT_MODE == "複合体から自動抽出":
    _counts = _contacts_with_partners()
    if not _counts:
        print("\n相手鎖が見つかりません。ステップ5 で KEEP_CHAINS に複合体の一部を指定するか、")
        print("HOTSPOT_MODE を「手動で番号を指定」にしてください。")
    else:
        _ranked = sorted(_counts, key=lambda k: (-_counts[k], k))
        _partners = ", ".join(globals().get("DROPPED_CHAINS", []))
        print(f"\n外した鎖 {_partners} と接していた残基: {len(_counts)} 個")
        if SHOW_CONTACTS:
            print("  接触原子数の多い順（上位 12）")
            for _num in _ranked[:12]:
                _r = by_auth.get(_num)
                _tag = f"{_one(_r)}{_num}" if _r is not None else f"?{_num}"
                print(f"    {_tag:>8}  原子 {_counts[_num]:>3} 個  label {auth2label.get(_num)}")
        _wanted = _spread_out(_ranked, N_HOTSPOTS)
        print(f"\n  互いに 7 Å 以上離れた上位 {len(_wanted)} 残基を選びました: "
              + ", ".join(f"{_one(by_auth[n])}{n}" for n in _wanted))
        print("  別の面を狙う場合は、上の一覧から番号を選んで手動指定に切り替えてください。")
elif HOTSPOT_MODE == "手動で番号を指定":
    _wanted = [int(x) for x in HOTSPOTS_AUTH.replace(" ", "").split(",") if x]
    assert _wanted, "HOTSPOTS_AUTH に著者番号を書いてください。"

HOTSPOT_SELECT = ""
HOTSPOT_LABELS = []

if _wanted:
    _absent = [w for w in _wanted if w not in auth2label]
    assert not _absent, f"残基 {_absent} は鎖 {TARGET_CHAIN} にありません。"
    HOTSPOT_LABELS = sorted(auth2label[w] for w in _wanted)

    print("\nホットスポット")
    _parts = []
    for w in _wanted:
        lab = auth2label[w]
        res = by_label[lab]
        if HOTSPOT_LEVEL == "residue":
            print(f"  {_one(res)}{w}（著者番号） -> label {lab}  [残基全体]")
            continue
        # 実際に座標がある原子だけ使う。欠けた側鎖を指定すると何にも当たらない選択式になる。
        atoms = [a for a in TIP_ATOMS.get(res.name, []) if res.find_atom(a, "*") is not None]
        if not atoms:
            fb = res.find_atom("CB", "*") or res.find_atom("CA", "*")
            atoms = [fb.name] if fb is not None else []
            print(f"  {_one(res)}{w}（著者番号） -> label {lab}  "
                  f"[先端原子なし、{atoms or '該当なし'} で代用]")
        else:
            print(f"  {_one(res)}{w}（著者番号） -> label {lab}  [{','.join(atoms)}]")
        if atoms:
            _parts.append(f"sel('{TARGET_CHAIN}/*/{lab}/[{','.join(atoms)}]')")

    if HOTSPOT_LEVEL == "residue" or not _parts:
        HOTSPOT_SELECT = f"sel('{TARGET_CHAIN}/*/{HOTSPOT_LABELS}')"
    elif len(_parts) == 1:
        HOTSPOT_SELECT = _parts[0]
    else:
        HOTSPOT_SELECT = "(" + " or ".join(_parts) + ")"

    print("\nステップ7 が使う選択式")
    print(" ", HOTSPOT_SELECT)
else:
    print("\nホットスポット指定なし。標的表面のどこを狙うかはモデルに任せます。")

---
## ステップ7 · 設計条件を書く

設計 1 件は短い YAML で表します。要素は 3 つです。
`input` が標的構造、`contig` が「どこを残し、どこに設計領域を挿すか」、
`conditions` が「モデルに何を守らせるか」です。

contig に書くトークンの意味は次のとおりです。

| トークン | 意味 |
|---|---|
| `B1-118` | 鎖 B の label 番号 1 から 118 までをそのまま使う |
| `/0` | 鎖の区切り。これ以降は別の鎖になる |
| `BINDER:55-85` | 名前付きの設計領域。長さは 55 から 85 の間で毎回抽選される |
| `20-30` | 名前なしの設計領域 |

conditions で使う条件は次のとおりです。

| 条件 | 意味 |
|---|---|
| `C_CRDGRP` | 指定した原子群の相対位置を保つ。全体の向きと側鎖は自由 |
| `C_CRD` | 入力ファイルの絶対座標に固定する |
| `C_SEQ` | 残基の種類を保つ |
| `C_HOT` | ホットスポットとして狙わせる |
| `C_SEQ_PROB_CYS` | システインの出現確率を指定する（0.0 で禁止） |

重要な注意があります。書かなかった条件はモデルに渡りません。
標的に `C_SEQ` を付けないと、標的の配列までバインダーと一緒に設計し直されます。
`TARGET_MODE` はそのための既定の与え方で、`rigid` は標的の主鎖形状を 1 つの剛体として保ち、
全体の向きと側鎖の自由度を残します。`fixed` は入力の座標に完全に固定します。
ふつうは `rigid` を使ってください。

`EXCLUDE_CYS` はシステインを禁じます。分泌タンパク質として作るなら既定のオンが扱いやすく、
ジスルフィド結合を使いたい場合はオフにします。

さらに細かい条件（埋没度の `C_RAS`、水素結合の `C_HB`、トポロジーの `C_CATH`、複数設計の一括指定など）が
必要な場合は、このセルのコードを表示して `RAW_YAML` に直接書き、`USE_RAW_YAML` をオンにしてください。
文法は[公式ドキュメント](https://rfd4-proteina.org/latest/reference/yaml-json.html)にあります。

In [ ]:
#@title 7. design.yaml の作成
import os, yaml, gemmi

BINDER_MIN_LEN = 55  #@param {type:"slider", min:20, max:200, step:5}
BINDER_MAX_LEN = 85  #@param {type:"slider", min:20, max:300, step:5}
TARGET_MODE    = "rigid"  #@param ["rigid", "fixed"]
EXCLUDE_CYS    = True  #@param {type:"boolean"}
USE_RAW_YAML   = False  #@param {type:"boolean"}
#  USE_RAW_YAML: 下の RAW_YAML をそのまま使います（上の設定は無視されます）。

RUN_DIR    = globals().get("RUN_DIR", "/content/run")
TARGET_CIF = globals().get("TARGET_CIF")
SPEC_PATH  = os.path.join(RUN_DIR, "design.yaml")
RUN_TAG    = globals().get("RUN_TAG", "design")
assert TARGET_CIF, "先にステップ5 を実行してください。"
assert BINDER_MAX_LEN >= BINDER_MIN_LEN, "BINDER_MAX_LEN が BINDER_MIN_LEN より小さいです。"

# 上級者向け。USE_RAW_YAML をオンにすると、こちらが使われます。
# __TARGET__ はステップ5 で書き出した標的のファイル名に置き換わります。
# 鎖の範囲（B1-118）とホットスポットの番号は label 番号なので、
# ステップ5 とステップ6 が表示した値に合わせて書き換えてください。
RAW_YAML = r"""
short_binder:
  input: ./inputs/__TARGET__
  contig: "BINDER:45-70, /0, B1-118"
  conditions:
    C_CRDGRP:
      1:
        - select: "sel('B/*/*/[N,CA,C,O]') and not has_nan_coord()"
    C_SEQ:
      true:
        - select: "sel('B')"
    C_HOT:
      true:
        - select: "sel('B/*/[34, 94, 102]')"

long_binder:
  input: ./inputs/__TARGET__
  contig: "BINDER:90-120, /0, B1-118"
  conditions:
    C_CRDGRP:
      1:
        - select: "sel('B/*/*/[N,CA,C,O]') and not has_nan_coord()"
    C_SEQ:
      true:
        - select: "sel('B')"
"""

def _chain_ranges(cif_path):
    """鎖ごとに label 番号の範囲を contig トークンにする。"""
    st = gemmi.read_structure(cif_path); st.setup_entities()
    out = []
    for ch in st[0]:
        labels = [r.label_seq for r in ch if r.label_seq is not None]
        if labels:
            out.append((ch.name, min(labels), max(labels)))
    return out

if USE_RAW_YAML:
    text = RAW_YAML.replace("__TARGET__", os.path.basename(TARGET_CIF))
    assert "__TARGET__" not in text, "標的が読み込まれていません。"
    spec = yaml.safe_load(text)
    assert isinstance(spec, dict) and spec, "YAML が設計の辞書として読めません。"
    with open(SPEC_PATH, "w") as fh:
        fh.write(text.strip() + "\n")
else:
    ranges = _chain_ranges(TARGET_CIF)
    chains = [c for c, _, _ in ranges]

    # 設計領域を先に置き、鎖の区切りを挟んで標的を続ける。
    contig = f"BINDER:{BINDER_MIN_LEN}-{BINDER_MAX_LEN}, /0, " + \
             ", /0, ".join(f"{c}{lo}-{hi}" for c, lo, hi in ranges)

    def _any_of(items):
        return items[0] if len(items) == 1 else "(" + " or ".join(items) + ")"

    sel_seq = _any_of([f"sel('{c}')" for c in chains])
    sel_bb  = _any_of([f"sel('{c}/*/*/[N,CA,C,O]')" for c in chains])

    conditions = {}
    if TARGET_MODE == "rigid":
        # 主鎖を 1 つの剛体グループとして扱う。全体の向きと側鎖は自由に残る。
        conditions["C_CRDGRP"] = {1: [{"select": f"{sel_bb} and not has_nan_coord()"}]}
    else:
        conditions["C_CRD"] = {True: [{"select": f"{sel_bb} and not has_nan_coord()"}]}
    conditions["C_SEQ"] = {True: [{"select": sel_seq}]}

    hot = globals().get("HOTSPOT_SELECT", "")
    if hot:
        conditions["C_HOT"] = {True: [{"select": hot}]}
    if EXCLUDE_CYS:
        conditions["C_SEQ_PROB_CYS"] = {0.0: [{"select": "res_name == 'BINDER'"}]}

    spec = {RUN_TAG: {"input": f"./inputs/{os.path.basename(TARGET_CIF)}",
                      "contig": contig,
                      "conditions": conditions}}
    with open(SPEC_PATH, "w") as fh:
        yaml.safe_dump(spec, fh, sort_keys=False, default_flow_style=False, allow_unicode=True)

# ステップ8 が使う情報
SPEC_NAMES      = [k for k in spec if k != "global_args"]
SPEC_CONDITIONS = set()
SPEC_MULTICHAIN = False
for _name in SPEC_NAMES:
    _body = spec[_name]
    if not isinstance(_body, dict):
        continue
    SPEC_CONDITIONS |= {str(k) for k in (_body.get("conditions") or {})}
    if "/0" in str(_body.get("contig", "")):
        SPEC_MULTICHAIN = True

print("設計の数   :", len(SPEC_NAMES), SPEC_NAMES)
print("条件       :", ", ".join(sorted(SPEC_CONDITIONS)) or "なし")
print("書き出し先 :", SPEC_PATH, "\n")
print(open(SPEC_PATH).read())

---
## ステップ8 · 生成の実行

`RECYCLE_STRATEGY` は思考量です。どの設定でも積分ステップ数は 60 で固定され、
内部の再帰的な精緻化を何回まわすかが変わります。
`thinking_low` から `thinking_high` へ上げると成功率は上がり、処理速度は 2 倍以上遅くなります。
T4 では `thinking_low` から始めてください。

積分ステップ数は 60 で固定しており、変更するつまみを置いていません。
スケジュールが 60 ステップ分として定義されているため、値がずれると生成の直前で停止します。
どうしても変えたい場合だけ `EXTRA_OVERRIDES` に
`generation.args.nsteps=<値>` と書けますが、品質が落ちるので勧めません。

`PRESET` は思考量、設計本数、バッチサイズをまとめて決めます。
個別に触りたい場合は `手動` を選んでください。
この3つを別々に動かすと組み合わせが壊れやすいので、既定では連動させています。

セルの冒頭でプリフライトが走り、次の点を実行前に確認します。
実行していないステップ、再接続による GPU の変更、標的より古い設計条件、
条件を変えた設計が同じ実験に混ざること、そして残基数とメモリの釣り合いです。
問題が見つかった場合は、長い生成を始める前に止まります。

`NUM_DESIGNS` は設計 1 件あたりの生成本数です。総数は `NUM_DESIGNS × 設計の数` になります。
バインダー設計の歩留まりは高くないので、本番では数百本単位で回し、再検証で絞り込みます。
このチュートリアルの既定値は 4 本です。

`HOTSPOT_SAMPLING` は多様性のつまみです。
オフだと全ての本が同じ原子を狙いますが、オンにすると 1 本ごとに指定した周囲から 2 個か 3 個を引き直すので、
同じエピトープの別の攻め方が出てきます。

計算が終わるまでログが流れます。`skip_existing=true` のため、途中で切断しても、
ステップ 2 で同じ `RUN_TAG` を指定してからこのセルを再実行すれば、足りない分だけ追加生成されます。

In [ ]:
#@title 8. 生成の実行
import os, glob, json, hashlib, subprocess, time

PRESET = "GPUに合わせる"  #@param ["GPUに合わせる", "T4向け・軽量", "標準", "高品質", "手動"]
#  PRESET: 思考量、設計本数、バッチサイズをまとめて決めます。
#          「手動」を選んだときだけ、下の3つの値がそのまま使われます。
ALLOW_SPEC_CHANGE = False  #@param {type:"boolean"}
#  ALLOW_SPEC_CHANGE: 条件を変えた設計を同じ実験に混ぜることを許します。
#                     ふだんはオフのままで、条件を変えたら RUN_TAG を変えてください。

NUM_DESIGNS      = 4  #@param {type:"slider", min:1, max:64, step:1}
BATCH_SIZE       = 1  #@param {type:"slider", min:1, max:8, step:1}
RECYCLE_STRATEGY = "thinking_low"  #@param ["thinking_low", "thinking_medium", "thinking_high", "low/front_loaded", "medium/front_loaded", "high/front_loaded", "low/flat", "null"]
GUIDANCE_W       = 1.0  #@param {type:"number"}
#  GUIDANCE_W: 条件付けの強さ。1.0 で無効です。
SEED             = 0    #@param {type:"integer"}
#  SEED: -1 にすると毎回違う設計が出ます。
HOTSPOT_SAMPLING = False  #@param {type:"boolean"}
HOTSPOT_PATCH_A  = 15.0   #@param {type:"number"}
#  HOTSPOT_PATCH_A: ホットスポットを引き直す範囲の直径です。大きいほどエピトープがずれます。
EXTRA_OVERRIDES  = ""  #@param {type:"string"}
#  EXTRA_OVERRIDES: Hydra の key=value を空白区切りで追記します（最後に適用されます）。
VERBOSE          = True  #@param {type:"boolean"}

# 積分ステップ数は 60 で固定する。思考量のスケジュールが 60 ステップ分として
# 定義されており、値がずれると生成の直前で ValueError になる。変更するつまみは置かない。
NSTEPS = 60

# ---- プリセットの適用 ----
PRESETS = {
    "T4向け・軽量": {"recycle": "thinking_low",    "designs": 4, "batch": 1},
    "標準":         {"recycle": "thinking_medium", "designs": 8, "batch": 1},
    "高品質":       {"recycle": "thinking_high",   "designs": 8, "batch": 1},
}
if PRESET == "GPUに合わせる":
    _p = globals().get("SUGGEST")
    assert _p, "ステップ1 を実行してください。GPU に応じた推奨値がありません。"
elif PRESET == "手動":
    _p = None
else:
    _p = PRESETS[PRESET]

if _p:
    RECYCLE_STRATEGY, NUM_DESIGNS, BATCH_SIZE = _p["recycle"], _p["designs"], _p["batch"]
    print(f"プリセット「{PRESET}」を適用しました: "
          f"{RECYCLE_STRATEGY} / 設計 {NUM_DESIGNS} 本 / バッチ {BATCH_SIZE}\n")

# ---- プリフライト: 実行前に設定の整合性を確認する ----
_need = {"GPU_NAME": 1, "PRECISION": 1, "KERNEL_OVERRIDES": 1, "RUN_DIR": 2,
         "GEN_BIN": 3, "RFD4_ENV": 3, "TARGET_CIF": 5, "SPEC_CONDITIONS": 7}
_missing = [f"ステップ{s}（{n} がありません）" for n, s in _need.items() if n not in globals()]
assert not _missing, "実行していないステップがあります: " + "、".join(_missing)

RUN_DIR    = RUN_DIR
SPEC_PATH  = os.path.join(RUN_DIR, "design.yaml")
TARGET_CIF = TARGET_CIF
conds      = SPEC_CONDITIONS
problems, cautions = [], []

if not os.path.exists(SPEC_PATH):
    problems.append("design.yaml がありません。ステップ7 を実行してください。")
if not os.path.exists(GEN_BIN):
    problems.append("rfd4-generate がありません。ステップ3 を実行してください。")

# 再接続で GPU が変わると、ステップ1 が決めた精度とカーネル設定が合わなくなる
_now = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
if _now and _now[0].strip() != GPU_NAME:
    problems.append(f"GPU が {GPU_NAME} から {_now[0].strip()} に変わっています。"
                    "ステップ1 から実行し直してください。")

# 標的を作り直したあとに設計条件を更新していないと、contig が古い番号を指す
if (os.path.exists(SPEC_PATH) and os.path.exists(TARGET_CIF)
        and os.path.getmtime(SPEC_PATH) < os.path.getmtime(TARGET_CIF)):
    problems.append("design.yaml が target.cif より古いです。ステップ7 を実行し直してください。")

if globals().get("HF_HOME", "").startswith("/content/drive"):
    problems.append("重みが Drive 上にあります。safetensors の読み込みが壊れるため、"
                    "ステップ2 とステップ4 を実行し直してください。")

if HOTSPOT_SAMPLING and "C_HOT" not in conds:
    problems.append("ホットスポットの引き直しには C_HOT が必要です。ステップ6 で指定してください。")

# 条件の違う設計が同じ実験に混ざると、ステップ9 の表が別物の寄せ集めになる
_spec_hash = (hashlib.sha1(open(SPEC_PATH, "rb").read()).hexdigest()[:10]
              if os.path.exists(SPEC_PATH) else "")
_state_path = os.path.join(RUN_DIR, "run_state.json")
_state = json.load(open(_state_path)) if os.path.exists(_state_path) else {}
_have = sorted(glob.glob(f"{RUN_DIR}/inference_samples/**/*_generated.cif*", recursive=True))
if _have and _state.get("spec_hash", _spec_hash) != _spec_hash and not ALLOW_SPEC_CHANGE:
    problems.append(
        f"この実験には別の条件で作った設計が {len(_have)} 本あります。"
        "ステップ2 で RUN_TAG を変えるか、ALLOW_SPEC_CHANGE をオンにしてください。")

# メモリの目安。実測ではなく、空きメモリに対する残基数の経験的な上限。
_tokens = sum(len(s) for s in globals().get("TARGET_CHAIN_SEQ", {}).values()) \
          + globals().get("BINDER_MAX_LEN", 0)
_budget = int(14 * globals().get("GPU_MEM_GB", 15))
if _tokens * BATCH_SIZE > _budget and BATCH_SIZE > 1:
    cautions.append(f"合計 {_tokens} 残基 × バッチ {BATCH_SIZE} は "
                    f"{GPU_MEM_GB} GB に対して大きいため、バッチを 1 に下げました。")
    BATCH_SIZE = 1
if _tokens > _budget:
    cautions.append(f"合計 {_tokens} 残基は {GPU_MEM_GB} GB では重い可能性があります"
                    f"（目安 {_budget} 残基）。メモリ不足になる場合は標的を切り詰めるか、"
                    "ステップ7 で BINDER_MAX_LEN を下げてください。")

for c in cautions:
    print("注意:", c)
assert not problems, "設定を確認してください:\n  - " + "\n  - ".join(problems)
print("プリフライト: 問題なし" if not cautions else "プリフライト: 上の注意を確認してください")
print()

overrides = [
    f"inputs={SPEC_PATH}",
    f"out_dir={RUN_DIR}",
    f"num_replicates={NUM_DESIGNS}",
    f"batch_size={BATCH_SIZE}",
    f"recycle_strategy={RECYCLE_STRATEGY}",
    f"generation.args.nsteps={NSTEPS}",
    f"generation.args.guidance_w={GUIDANCE_W}",
    f"precision={globals().get('PRECISION', 'bf16-mixed')}",
    "skip_existing=true",        # 既存の設計を残し、足りない分だけ作る
    "prevalidate_inputs=true",   # contig の誤りをモデル読み込み前に検出する
    "save_chain_of_interest=true",
    "compile_lengths=null",      # torch.compile の暖機は短い実行では割に合わない
    # 事前フィルタは「再検証に送る価値があるか」を判断する仕組みで、既定では
    # 不合格の構造ファイルを削除する。再検証をここでは行わないので、全部残す。
    "delete_failing_pre_oracle=false",
]
overrides += globals().get("KERNEL_OVERRIDES", [])
if SEED >= 0:
    overrides.append(f"seed={SEED}")

# 条件に応じて、オラクル不要の指標だけを有効にする
tags = ["design"]
if "C_HOT" in conds:
    tags.append("hotspot_conditioning")
if "C_CRDGRP" in conds:
    tags.append("internal_coordinate_conditioning")
if "C_CRD" in conds:
    tags.append("3d_coordinate_conditioning")
if globals().get("SPEC_MULTICHAIN"):
    tags.append("interchain_interactions")
overrides.append("metrics_tags=[" + ",".join(dict.fromkeys(tags)) + "]")

# ホットスポットの引き直しは CLI では指定できないので、設定ファイルを合成する
EXPERIMENT_CFG = None
if HOTSPOT_SAMPLING:
    assert "C_HOT" in conds, "ホットスポットの引き直しには C_HOT が必要です。ステップ6 で指定してください。"
    EXPERIMENT_CFG = os.path.join(RUN_DIR, "sampling.yaml")
    with open(EXPERIMENT_CFG, "w") as fh:
        fh.write(
            "# このノートブックが生成した設定ファイル。\n"
            "defaults:\n  - inference\n  - _self_\n\n"
            "hydra:\n  searchpath:\n    - file://${oc.env:PROJECT_ROOT}/configs\n\n"
            "generation:\n  dataset:\n    sampling_transforms:\n"
            "      - _target_: rfproteina.transforms.hotspot_sampling.SampleNearbyHotspots\n"
            "        num_hotspots: [2, 3]\n"
            "        min_distance: 5.0\n"
            f"        patch_diameter: {HOTSPOT_PATCH_A}\n"
            "        min_atom_sasa: 1.0\n"
            "        shape: sphere\n")

if EXTRA_OVERRIDES.strip():
    overrides += EXTRA_OVERRIDES.split()

cmd = [GEN_BIN]
if EXPERIMENT_CFG:
    cmd += ["--config-path", RUN_DIR, "--config-name", "sampling"]
cmd += overrides

env = dict(globals().get("RFD4_ENV", os.environ))
env["MPLBACKEND"] = "Agg"
env["HF_HOME"] = globals().get("HF_HOME", env.get("HF_HOME", "/content/weights"))
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # 断片化による OOM を減らす

LOG_PATH = os.path.join(RUN_DIR, "generate.log")
print("有効な指標:", ",".join(dict.fromkeys(tags)))
print("\n+ " + " ".join(cmd) + "\n", flush=True)

_t0 = time.time()
_keep = ("error", "traceback", "failed", "exception", "out of memory", "warning")
_show = ("Generat", "Sampling", "Saved", "Wrote", "Metric", "example", "Seed", "Completed")
with open(LOG_PATH, "w") as log:
    proc = subprocess.Popen(cmd, cwd=RUN_DIR, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        low = line.lower()
        if (VERBOSE or any(k in low for k in _keep) or any(k in line for k in _show)
                or "%|" in line or "it/s" in line or "s/it" in line or "INFO" not in line):
            print(line, end="")
    proc.wait()

_made = sorted(glob.glob(f"{RUN_DIR}/inference_samples/**/*_generated.cif*", recursive=True))

# 何の条件で作った設計なのかを実験フォルダに残す。プリフライトもこれを読む。
if _made:
    with open(_state_path, "w") as fh:
        json.dump({"spec_hash": _spec_hash, "preset": PRESET,
                   "recycle_strategy": RECYCLE_STRATEGY, "nsteps": NSTEPS,
                   "num_designs": NUM_DESIGNS, "batch_size": BATCH_SIZE,
                   "guidance_w": GUIDANCE_W, "seed": SEED,
                   "binder_len": [globals().get("BINDER_MIN_LEN"),
                                  globals().get("BINDER_MAX_LEN")],
                   "hotspot_sampling": HOTSPOT_SAMPLING,
                   "precision": PRECISION, "gpu": GPU_NAME,
                   "metrics_tags": ",".join(dict.fromkeys(tags))},
                  fh, ensure_ascii=False, indent=1)

print(f"\n経過時間 : {time.time() - _t0:.0f} 秒")
print(f"設計数   : {len(_made)} 本（この実験の累計）")
print(f"ログ     : {LOG_PATH}")
if proc.returncode != 0 and not _made:
    print("\n失敗しました。よくある原因は次の3つです。")
    print("  1. メモリ不足  -> BATCH_SIZE=1、バインダー長を短く、thinking_low に下げる")
    print("  2. contig の誤り -> ステップ7 の出力と標的の label 番号を見比べる")
    print("  3. 重みの取得失敗 -> ステップ4 を再実行する")

---
## ステップ9 · 結果を一覧にする

生成された構造を読み、設計 1 本につき 1 行の表を作ります。
設計された鎖は、標的の配列と一致しない鎖として見分けます。鎖の名前は入力と変わることがあるためです。

表に載る値は、長さ、標的と接触している残基数、慣性半径、疎水性残基の割合、システイン数、
そしてステップ 8 の指標ファイルから結合した列です。

これらは検証ではありません。再検証に送る価値があるかどうかを安く見積もるための目安です。
特に次の 2 つは判断に使えます。
`clash` 系の列は原子の衝突、`sequence_adherence` 系の列は条件付けた配列を再現できたかを表します。
バインダー設計で後者に違反が出ている場合は、標的の配列まで作り直されている可能性があります。

In [ ]:
#@title 9. 設計の一覧表と分布図
import os, glob, math
import numpy as np, pandas as pd, gemmi, matplotlib.pyplot as plt
from IPython.display import display

CONTACT_CUTOFF_A = 5.0  #@param {type:"number"}
#  CONTACT_CUTOFF_A: 設計残基の CB（Gly は CA）が標的の重原子からこの距離以内なら界面残基とみなします。
SHOW_PLOTS = True  #@param {type:"boolean"}

RUN_DIR = globals().get("RUN_DIR", "/content/run")
RUN_TAG = globals().get("RUN_TAG", "design")
# ステップ5 で選んだ標的の鎖名。contig はこの名前をそのまま使っているので、
# 生成された複合体でもこの名前の鎖が標的であり、残りがデザイン（バインダー）になる。
TARGET_CHAIN_NAMES = set(globals().get("TARGET_CHAIN_SEQ", {}).keys())

cifs = sorted(glob.glob(f"{RUN_DIR}/inference_samples/**/*_generated.cif*", recursive=True))
assert cifs, f"{RUN_DIR}/inference_samples に設計がありません。ステップ8 を確認してください。"

# どの条件で作った設計かをステップ8 が残しているので、表の前に示す
_state_file = os.path.join(RUN_DIR, "run_state.json")
if os.path.exists(_state_file):
    import json as _json
    _s = _json.load(open(_state_file))
    print(f"条件: {_s.get('recycle_strategy')} / バインダー {_s.get('binder_len')} 残基 / "
          f"シード {_s.get('seed')} / {_s.get('gpu')}")

HYDROPHOBIC = set("AVLIMFWYC")

rows, STRUCTURES = [], {}
for path in cifs:
    try:
        st = gemmi.read_structure(path)
        st.setup_entities(); st.remove_hydrogens()
        # 結晶格子を外す。残すと対称の写しを接触として数えてしまう。
        st.cell = gemmi.UnitCell(); st.spacegroup_hm = "P 1"
        st.setup_cell_images()
    except Exception as exc:
        print("読み飛ばし:", os.path.basename(path), "-", exc)
        continue

    design_id = os.path.basename(path).split("_generated")[0]
    STRUCTURES[design_id] = path

    # 鎖名で判定する。contig がステップ5 の標的の鎖名をそのまま使っているため、
    # 配列や長さを見ずに名前の一致だけで標的とデザイン（バインダー）を分けられる。
    # （標的は rigid/fixed 条件下でも欠損残基を埋めて出力されることがあり、
    #   配列や長さで判定すると標的がデザイン側に混入しうる。）
    tgt_names, des_res = [], []
    for ch in st[0]:
        if ch.name in TARGET_CHAIN_NAMES:
            tgt_names.append(ch.name)
        else:
            des_res += list(ch)
    if not des_res:                       # 標的が見分けられない場合は全体を設計として扱う
        des_res = [r for ch in st[0] for r in ch]
        tgt_names = []

    ca = np.array([a.pos.tolist() for a in
                   (r.find_atom("CA", "*") for r in des_res) if a is not None])
    rg = float(np.sqrt(((ca - ca.mean(0)) ** 2).sum(1).mean())) if len(ca) else math.nan
    seq = gemmi.one_letter_code([r.name for r in des_res]).upper()

    n_iface = 0
    if tgt_names:
        ns = gemmi.NeighborSearch(st, CONTACT_CUTOFF_A).populate()
        tgt_set = set(tgt_names)
        for r in des_res:
            at = r.find_atom("CB", "*") or r.find_atom("CA", "*")
            if at is None:
                continue
            marks = ns.find_atoms(at.pos, "\0", radius=CONTACT_CUTOFF_A)
            if any(st[0][m.chain_idx].name in tgt_set for m in marks):
                n_iface += 1

    rows.append({
        "design_id": design_id,
        "design_len": len(des_res),
        "interface_res": n_iface,
        "frac_interface": round(n_iface / max(len(des_res), 1), 3),
        "radius_gyration_A": None if math.isnan(rg) else round(rg, 2),
        "frac_hydrophobic": round(sum(c in HYDROPHOBIC for c in seq) / max(len(seq), 1), 3),
        "n_cys": seq.count("C"),
        "sequence": seq,
        "path": path,
    })

df = pd.DataFrame(rows).sort_values(["interface_res", "design_len"],
                                    ascending=[False, True]).reset_index(drop=True)

# ステップ8 が書いた指標を design_id で結合する
_mets = sorted(glob.glob(f"{RUN_DIR}/inference_metrics/per_example_metrics*.csv"))
if _mets:
    mdf = pd.concat([pd.read_csv(m) for m in _mets], ignore_index=True)
    key = next((c for c in mdf.columns
                if c.lower() in ("example_id", "design_id", "name", "id", "sample_id")), None)
    if key:
        drop = [c for c in mdf.columns if c.startswith("generation.")
                or c in ("ckpt_path", "ckpt_name", "seed")]
        df = df.merge(mdf.drop(columns=drop).rename(columns={key: "design_id"}),
                      on="design_id", how="left")
        print(f"指標 {len(mdf)} 行を結合しました。")

TABLE_PATH = os.path.join(RUN_DIR, f"{RUN_TAG}_designs.csv")
df.to_csv(TABLE_PATH, index=False)

# 判断に効く列を前に出す
front = ["design_id", "design_len", "interface_res", "frac_interface"]
front += [c for c in df.columns if "clash" in c.lower()]
front += [c for c in df.columns if any(k in c.lower() for k in
          ("adherence", "hotspot", "rmsd"))]
front += ["radius_gyration_A", "frac_hydrophobic", "n_cys"]
front = [c for c in dict.fromkeys(front) if c in df.columns]
rest = [c for c in df.columns if c not in front + ["sequence", "path"]]

pd.set_option("display.max_columns", 60)
print(f"\n設計 {len(df)} 本。表を保存しました: {TABLE_PATH}\n")
display(df[front + rest].head(30))

print(f"\n長さ       : {df['design_len'].min()}-{df['design_len'].max()} 残基"
      f"（中央値 {int(df['design_len'].median())}）")

_lo, _hi = globals().get("BINDER_MIN_LEN"), globals().get("BINDER_MAX_LEN")
if _lo and _hi:
    _bad = df[(df["design_len"] < _lo) | (df["design_len"] > _hi)]
    if len(_bad):
        print(f"\n注意: {len(_bad)} 本が指定範囲（{_lo}-{_hi} 残基）から外れています。")
        print("      鎖名での判定が外れている可能性があります。")
        print("      ステップ5 の実行と、生成構造の鎖名を確認してください。")
if df["interface_res"].max() > 0:
    print(f"界面残基数 : {df['interface_res'].min()}-{df['interface_res'].max()}"
          f"（中央値 {int(df['interface_res'].median())}）")
    print("             設計されたタンパク質間相互作用では 15-30 残基程度が標準的です。")
    print("             極端に少ない場合は標的から離れており、多すぎる場合は")
    print("             狙ったエピトープではなく切り出しの端に巻き付いていることがあります。")
print("\nここまでの値は検証ではありません。折り畳みと結合は再検証で確かめてください。")

if SHOW_PLOTS and len(df) > 1:
    cols = [c for c in ("design_len", "interface_res", "radius_gyration_A", "frac_hydrophobic")
            if c in df.columns]
    fig, axes = plt.subplots(1, len(cols), figsize=(3.3 * len(cols), 2.7), squeeze=False)
    for ax, col in zip(axes[0], cols):
        vals = pd.to_numeric(df[col], errors="coerce").dropna().values
        if not len(vals):
            ax.axis("off"); continue
        ax.hist(vals, bins=min(15, max(4, len(vals) // 2)), color="#3f6f9f", edgecolor="white")
        ax.axvline(np.median(vals), color="#b03a2e", ls="--", lw=1.2)
        ax.set_xlabel(col.replace("_", " ")); ax.set_ylabel("count")
    fig.suptitle(f"{RUN_TAG} -- {len(df)} designs (dashed line = median)", y=1.04)
    plt.tight_layout()
    _fig_dir = os.path.join(RUN_DIR, "figures"); os.makedirs(_fig_dir, exist_ok=True)
    plt.savefig(os.path.join(_fig_dir, f"{RUN_TAG}_distribution.png"), dpi=130,
                bbox_inches="tight")
    plt.show()

---
## ステップ10 · 立体構造を見る

標的を灰色、設計されたバインダーを虹色で表示します。
`RANK` はステップ 9 の表の順位で、1 が界面残基数の最も多い設計です。

見るべき点は 2 つあります。
バインダーが狙ったクレフトに入っているか、そして二次構造がきちんと形成されているかです。
ほどけた紐のような設計や、標的の横に並んでいるだけの設計は、この時点で外せます。

In [ ]:
#@title 10. 3D ビューア
import py3Dmol, gemmi
from IPython.display import display

RANK  = 1  #@param {type:"integer"}
STYLE = "cartoon"  #@param ["cartoon", "stick", "sphere"]
SHOW_HOTSPOTS = True  #@param {type:"boolean"}

assert "df" in globals() and len(df), "先にステップ9 を実行してください。"
row = df.iloc[min(len(df), max(1, RANK)) - 1]
path = row["path"]
# 鎖名で判定する。ステップ9 と同じ規則にそろえる。
TARGET_CHAIN_NAMES = set(globals().get("TARGET_CHAIN_SEQ", {}).keys())

st = gemmi.read_structure(path); st.setup_entities()
tgt_chains = [c.name for c in st[0] if c.name in TARGET_CHAIN_NAMES]
des_chains = [c.name for c in st[0] if c.name not in TARGET_CHAIN_NAMES]

print(f"{row['design_id']}  |  {row['design_len']} 残基  |  "
      f"界面残基 {row['interface_res']}  |  疎水性 {row['frac_hydrophobic']}")
print("標的の鎖:", tgt_chains or "（判別できません）", " 設計の鎖:", des_chains)
print("配列:", row["sequence"])

view = py3Dmol.view(width=760, height=520)
with open(path) as fh:
    view.addModel(fh.read(), "cif")
view.setStyle({}, {STYLE: {"color": "spectrum"}})
for c in tgt_chains:
    view.setStyle({"chain": c}, {STYLE: {"color": "lightgray"}})

if SHOW_HOTSPOTS and tgt_chains and globals().get("HOTSPOT_LABELS"):
    # 生成後の構造では残基番号が振り直されている場合があるため、表示できないこともある
    view.addStyle({"chain": tgt_chains[0], "resi": globals()["HOTSPOT_LABELS"]},
                  {"stick": {"colorscheme": "redCarbon", "radius": 0.22}})

view.zoomTo()
view.show()
print("\nより詳しく見る場合は、次のファイルを ChimeraX や PyMOL で開いてください。")
print(" ", path)

---
## ステップ11 · 配列を書き出して再検証に渡す

RFD4-Proteina は配列と構造を同時に返します。したがって本当の絞り込みは、
得られた配列を別のモデルで折り畳み直し、設計モデルと比べる作業になります。

このセルは、設計 1 本につき「バインダーの配列」と「標的の配列」を並べた FASTA を書きます。
AlphaFold3、Protenix、OpenFold3 などのノートブックに渡せば、複合体として予測できます。

予測結果の見方は次のとおりです。
界面の信頼度（ipTM など）と、設計モデルへの RMSD の両方を見てください。
信頼度が高くても別の向きで結合している場合があるため、片方だけでは判断できません。

In [ ]:
#@title 11. FASTA の書き出し
import os

TOP_N = 0  #@param {type:"integer"}
#  TOP_N: ステップ9 の表の上位 N 本だけ書き出します。0 ですべてです。
INCLUDE_TARGET = True  #@param {type:"boolean"}
MIN_LEN = 30  #@param {type:"integer"}

assert "df" in globals() and len(df), "先にステップ9 を実行してください。"
RUN_DIR = globals().get("RUN_DIR", "/content/run")
RUN_TAG = globals().get("RUN_TAG", "design")
tgt = globals().get("TARGET_CHAIN_SEQ", {}) if INCLUDE_TARGET else {}

sub = df if TOP_N <= 0 else df.head(TOP_N)
FASTA_PATH = os.path.join(RUN_DIR, f"{RUN_TAG}_designs.fasta")

count = 0
with open(FASTA_PATH, "w") as fh:
    for _, r in sub.iterrows():
        if r["design_len"] < MIN_LEN:
            continue
        fh.write(f">{r['design_id']}|binder|{r['design_len']}aa\n{r['sequence']}\n")
        for name, seq in tgt.items():
            fh.write(f">{r['design_id']}|target_{name}|{len(seq)}aa\n{seq}\n")
        count += 1

print(f"{count} 本を書き出しました -> {FASTA_PATH}")
if tgt:
    print("同梱した標的の鎖:", ", ".join(f"{k}（{len(v)} 残基）" for k, v in tgt.items()))
print("\n--- 先頭 6 行 ---")
print("".join(open(FASTA_PATH).readlines()[:6]))

---
## ステップ12 · 結果の取り出し

Drive を接続している場合、出力はすでに Drive 上にあるので、このセルは内容の一覧だけ表示します。
接続していない場合は実験フォルダを ZIP にまとめてダウンロードします。

In [ ]:
#@title 12. 出力の確認とダウンロード
import os, shutil

RUN_DIR = globals().get("RUN_DIR", "/content/run")
RUN_TAG = globals().get("RUN_TAG", "design")

if RUN_DIR.startswith("/content/drive/"):
    print("出力は Drive に保存されています:", RUN_DIR, "\n")
    for root, _dirs, files in os.walk(RUN_DIR):
        depth = root.count(os.sep) - RUN_DIR.count(os.sep)
        if depth < 2 and files:
            print(f"  {os.path.relpath(root, RUN_DIR)}  （{len(files)} ファイル）")
else:
    zip_base = f"/content/{RUN_TAG}"
    shutil.make_archive(zip_base, "zip", RUN_DIR)
    zip_path = zip_base + ".zip"
    print(f"ZIP を作成しました: {zip_path}（{os.path.getsize(zip_path) / 1e6:.1f} MB）")
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as exc:
        print("自動ダウンロードに失敗しました。左のファイルパネルから取得してください。", exc)

---
## 付録 · つまずいたときの確認先

### 出力ファイルの場所

| パス | 内容 |
|---|---|
| `inference_samples/<id>/<id>_generated.cif` | 設計された複合体 |
| `inference_samples/<id>/<id>_input.cif` | モデルが実際に見た条件付け。contig の意図を確かめるときに開く |
| `inference_samples/<id>/<id>_chain.cif` | バインダーだけを切り出した構造 |
| `inference_metrics/per_example_metrics*.csv` | 構造だけで計算した指標 |
| `<RUN_TAG>_designs.csv` | ステップ9 で作った一覧表 |
| `<RUN_TAG>_designs.fasta` | ステップ11 で作った再検証用の配列 |
| `generate.log` | 生成時のログ全文 |

### メモリ不足（CUDA out of memory）

次の順に試してください。
`BATCH_SIZE` を 1 にする、標的を狙う領域だけに切り詰める、バインダー長の範囲を狭める、
`RECYCLE_STRATEGY` を `thinking_low` に下げる、の 4 段階です。
`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` はステップ 8 ですでに設定しています。

### SafetensorError: header too large で止まる

キャッシュにある重みが読めない状態です。原因は 2 つあります。
`HF_HOME` が Drive 上にある場合は、FUSE 経由の mmap 読み出しが壊れます。ローカルディスクに置いてください。
ダウンロードが途中で切れた場合も同じエラーになります。
どちらもステップ4 の `FORCE_REDOWNLOAD` をオンにして実行すれば直ります。

### recycle schedule expands to 60 steps というエラーで止まる

積分ステップ数と思考量のスケジュールが食い違っています。
ステップ8 は 60 で固定しているので、通常は起きません。
`EXTRA_OVERRIDES` に `generation.args.nsteps=` を書いた場合だけ発生します。その指定を外してください。

### 残基番号が合わない

contig と `sel(...)` は `label_seq_id` を使います。PDB のページに出ている著者番号ではありません。
ステップ 6 が変換と差分の表示を行うので、差が一定でないと表示された場合は欠損を疑ってください。

### 標的の配列まで変わってしまう

ステップ 7 の `C_SEQ` が標的の鎖を覆っているか確認してください。
書かなかった条件はモデルに渡らないため、条件のない標的は設計対象として扱われます。
`sequence_adherence` 系の指標に違反が出ている場合も同じ原因です。

### 再現性

`SEED` を固定し、ステップ 8 の `EXTRA_OVERRIDES` に
`generation.model.bb_ca.gt.mode=ode generation.model.local_latents.gt.mode=ode`
を入れると、同じ入力から同じ設計が出ます。
重みを固定したい場合は `hf_ckpt_revision=<コミットID>` も追記してください。

### 次に試すこと

1. `RUN_TAG` を変えて、バインダー長や思考量の違う実験を並べる
2. ステップ6 の接触一覧から別の残基を選び、PD-1 の違う面を狙った設計と比べる
3. ステップ 7 の `USE_RAW_YAML` で、長さの違う設計を 1 ファイルにまとめて一度に回す
4. 書き出した FASTA を AlphaFold3 や OpenFold3 のノートブックで折り畳み直し、
   界面の信頼度と RMSD で絞り込む
5. `HOTSPOT_SAMPLING` をオンにして、同じ面の攻め方を増やす

### 参考リンク

- [RFD4-Proteina のリポジトリ](https://github.com/RosettaCommons/RFD4-Proteina)
- [YAML の文法リファレンス](https://rfd4-proteina.org/latest/reference/yaml-json.html)
- [4ZQK（PD-1 と PD-L1 の複合体）](https://www.rcsb.org/structure/4ZQK)